# Sanskrit-to-English Neural Machine Translation
## Assignment 2: NLU Course

Custom Transformer-based Seq2Seq architecture for translating Sanskrit to English.

## 1. Installation & Dependencies

In [1]:
!pip install torch torchtext sentencepiece pandas numpy matplotlib tqdm nltk bert-score


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
import time
import math
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import sentencepiece as spm
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import nltk
from nltk.translate.bleu_score import corpus_bleu, sentence_bleu

nltk.download('punkt')
nltk.download('punkt_tab')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


## 2. Data Loading

In [ ]:
DATA_DIR = "data"

train_sa = pd.read_csv(os.path.join(DATA_DIR, "train_sa_10000.csv"))
train_en = pd.read_csv(os.path.join(DATA_DIR, "train_en_10000.csv"))
dev_sa = pd.read_csv(os.path.join(DATA_DIR, "dev_sa_1000.csv"))
dev_en = pd.read_csv(os.path.join(DATA_DIR, "dev_en_1000.csv"))
test_sa = pd.read_csv(os.path.join(DATA_DIR, "test_sa_1000.csv"))
test_en = pd.read_csv(os.path.join(DATA_DIR, "test_en_1000.csv"))

train_df = train_sa.merge(train_en, on="Source_id")
dev_df = dev_sa.merge(dev_en, on="Source_id")
test_df = test_sa.merge(test_en, on="Source_id")

print(f"Train: {len(train_df)}, Dev: {len(dev_df)}, Test: {len(test_df)}")
print(f"\nSample:")
print(train_df.head())

## 3. Preprocessing & Tokenization (SentencePiece BPE)

In [ ]:
os.makedirs("tokenizer", exist_ok=True)

sa_col = "Sentence_sa"
en_col = "Sentence_en"

print(f"Sanskrit column: {sa_col}")
print(f"English column: {en_col}")

with open("tokenizer/train_sa.txt", "w", encoding="utf-8") as f:
    for sent in train_df[sa_col].dropna():
        f.write(str(sent).strip() + "\n")

with open("tokenizer/train_en.txt", "w", encoding="utf-8") as f:
    for sent in train_df[en_col].dropna():
        f.write(str(sent).strip() + "\n")

In [ ]:
VOCAB_SIZE_SA = 4000
VOCAB_SIZE_EN = 4000

spm.SentencePieceTrainer.train(
    input="tokenizer/train_sa.txt",
    model_prefix="tokenizer/sp_sa",
    vocab_size=VOCAB_SIZE_SA,
    model_type="bpe",
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3,
    character_coverage=1.0,
    num_threads=4
)

spm.SentencePieceTrainer.train(
    input="tokenizer/train_en.txt",
    model_prefix="tokenizer/sp_en",
    vocab_size=VOCAB_SIZE_EN,
    model_type="bpe",
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3,
    character_coverage=1.0,
    num_threads=4
)

sp_sa = spm.SentencePieceProcessor()
sp_sa.load("tokenizer/sp_sa.model")

sp_en = spm.SentencePieceProcessor()
sp_en.load("tokenizer/sp_en.model")

print(f"Sanskrit vocab size: {sp_sa.get_piece_size()}")
print(f"English vocab size: {sp_en.get_piece_size()}")

sample = str(train_df[sa_col].iloc[0])
print(f"\nSample tokenization (Sanskrit): {sample}")
print(f"Tokens: {sp_sa.encode_as_pieces(sample)}")

## 4. Dataset & DataLoader

In [ ]:
PAD_IDX = 0
UNK_IDX = 1
BOS_IDX = 2
EOS_IDX = 3
MAX_LEN = 128


class TranslationDataset(Dataset):
    def __init__(self, df, sp_src, sp_tgt, src_col, tgt_col, max_len=MAX_LEN):
        self.pairs = []
        for _, row in df.iterrows():
            src = str(row[src_col]).strip()
            tgt = str(row[tgt_col]).strip()
            if src and tgt and src != 'nan' and tgt != 'nan':
                src_ids = sp_src.encode(src)[:max_len - 2]
                tgt_ids = sp_tgt.encode(tgt)[:max_len - 2]
                src_ids = [BOS_IDX] + src_ids + [EOS_IDX]
                tgt_ids = [BOS_IDX] + tgt_ids + [EOS_IDX]
                self.pairs.append((src_ids, tgt_ids))

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src, tgt = self.pairs[idx]
        return torch.tensor(src, dtype=torch.long), torch.tensor(tgt, dtype=torch.long)


def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)
    src_padded = pad_sequence(src_batch, batch_first=True, padding_value=PAD_IDX)
    tgt_padded = pad_sequence(tgt_batch, batch_first=True, padding_value=PAD_IDX)
    return src_padded, tgt_padded


train_dataset = TranslationDataset(train_df, sp_sa, sp_en, sa_col, en_col)
dev_dataset = TranslationDataset(dev_df, sp_sa, sp_en, sa_col, en_col)
test_dataset = TranslationDataset(test_df, sp_sa, sp_en, sa_col, en_col)

BATCH_SIZE = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=0, pin_memory=True)
dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=0, pin_memory=True)

print(f"Train batches: {len(train_loader)}, Dev batches: {len(dev_loader)}, Test batches: {len(test_loader)}")

## 5. Model Architecture: Custom Transformer Seq2Seq

Architecture details:
- **Encoder**: 4-layer Transformer encoder with multi-head self-attention
- **Decoder**: 4-layer Transformer decoder with masked self-attention + cross-attention
- **Positional Encoding**: Sinusoidal
- **Dimensions**: d_model=256, d_ff=512, heads=4
- **Regularization**: Dropout, Label Smoothing

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_k = d_model // n_heads
        self.n_heads = n_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)
        Q = self.W_q(query).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(key).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(value).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        attn = self.dropout(F.softmax(scores, dim=-1))
        context = torch.matmul(attn, V)
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, self.n_heads * self.d_k)
        return self.W_o(context)


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.linear2(self.dropout(F.relu(self.linear1(x))))


class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, src_mask=None):
        attn_out = self.self_attn(x, x, x, src_mask)
        x = self.norm1(x + self.dropout1(attn_out))
        ff_out = self.ff(x)
        x = self.norm2(x + self.dropout2(ff_out))
        return x


class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, x, enc_out, tgt_mask=None, src_mask=None):
        attn_out = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout1(attn_out))
        cross_out = self.cross_attn(x, enc_out, enc_out, src_mask)
        x = self.norm2(x + self.dropout2(cross_out))
        ff_out = self.ff(x)
        x = self.norm3(x + self.dropout3(ff_out))
        return x


class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, d_ff, dropout=0.1, max_len=512):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_IDX)
        self.pos_enc = PositionalEncoding(d_model, max_len, dropout)
        self.layers = nn.ModuleList([EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.d_model = d_model

    def forward(self, src, src_mask=None):
        x = self.embedding(src) * math.sqrt(self.d_model)
        x = self.pos_enc(x)
        for layer in self.layers:
            x = layer(x, src_mask)
        return self.norm(x)


class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, d_ff, dropout=0.1, max_len=512):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_IDX)
        self.pos_enc = PositionalEncoding(d_model, max_len, dropout)
        self.layers = nn.ModuleList([DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.d_model = d_model

    def forward(self, tgt, enc_out, tgt_mask=None, src_mask=None):
        x = self.embedding(tgt) * math.sqrt(self.d_model)
        x = self.pos_enc(x)
        for layer in self.layers:
            x = layer(x, enc_out, tgt_mask, src_mask)
        return self.norm(x)


class TransformerNMT(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=256, n_layers=4,
                 n_heads=4, d_ff=512, dropout=0.1, max_len=512):
        super().__init__()
        self.encoder = Encoder(src_vocab_size, d_model, n_layers, n_heads, d_ff, dropout, max_len)
        self.decoder = Decoder(tgt_vocab_size, d_model, n_layers, n_heads, d_ff, dropout, max_len)
        self.output_proj = nn.Linear(d_model, tgt_vocab_size)
        self._init_parameters()

    def _init_parameters(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def make_src_mask(self, src):
        return (src != PAD_IDX).unsqueeze(1).unsqueeze(2)

    def make_tgt_mask(self, tgt):
        batch_size, tgt_len = tgt.size()
        pad_mask = (tgt != PAD_IDX).unsqueeze(1).unsqueeze(2)
        causal_mask = torch.tril(torch.ones(tgt_len, tgt_len, device=tgt.device)).bool()
        causal_mask = causal_mask.unsqueeze(0).unsqueeze(1)
        return pad_mask & causal_mask

    def forward(self, src, tgt):
        src_mask = self.make_src_mask(src)
        tgt_mask = self.make_tgt_mask(tgt)
        enc_out = self.encoder(src, src_mask)
        dec_out = self.decoder(tgt, enc_out, tgt_mask, src_mask)
        return self.output_proj(dec_out)

    def encode(self, src):
        src_mask = self.make_src_mask(src)
        return self.encoder(src, src_mask), src_mask

    def decode_step(self, tgt, enc_out, src_mask):
        tgt_mask = self.make_tgt_mask(tgt)
        dec_out = self.decoder(tgt, enc_out, tgt_mask, src_mask)
        return self.output_proj(dec_out[:, -1, :])

In [ ]:
D_MODEL = 256
N_LAYERS = 4
N_HEADS = 4
D_FF = 1024
DROPOUT = 0.3

model = TransformerNMT(
    src_vocab_size=sp_sa.get_piece_size(),
    tgt_vocab_size=sp_en.get_piece_size(),
    d_model=D_MODEL,
    n_layers=N_LAYERS,
    n_heads=N_HEADS,
    d_ff=D_FF,
    dropout=DROPOUT,
    max_len=MAX_LEN
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## 6. Training

- **Optimizer**: AdamW with weight decay
- **Scheduler**: Warmup + Cosine Annealing
- **Loss**: Cross-entropy with label smoothing
- **Gradient clipping**: max norm 1.0

In [ ]:
class LabelSmoothingLoss(nn.Module):
    def __init__(self, vocab_size, padding_idx=0, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing
        self.vocab_size = vocab_size
        self.padding_idx = padding_idx
        self.confidence = 1.0 - smoothing

    def forward(self, logits, target):
        logits = logits.reshape(-1, self.vocab_size)
        target = target.reshape(-1)
        log_probs = F.log_softmax(logits, dim=-1)
        nll_loss = -log_probs.gather(dim=-1, index=target.unsqueeze(1)).squeeze(1)
        smooth_loss = -log_probs.mean(dim=-1)
        loss = self.confidence * nll_loss + self.smoothing * smooth_loss
        mask = (target != self.padding_idx).float()
        loss = (loss * mask).sum() / mask.sum()
        return loss


class WarmupCosineScheduler:
    def __init__(self, optimizer, d_model, warmup_steps=4000, max_steps=50000):
        self.optimizer = optimizer
        self.d_model = d_model
        self.warmup_steps = warmup_steps
        self.max_steps = max_steps
        self.step_num = 0
        self.peak_lr = (d_model * warmup_steps) ** (-0.5)

    def step(self):
        self.step_num += 1
        if self.step_num <= self.warmup_steps:
            lr = self.d_model ** (-0.5) * self.step_num * self.warmup_steps ** (-1.5)
        else:
            progress = (self.step_num - self.warmup_steps) / (self.max_steps - self.warmup_steps)
            lr = self.peak_lr * 0.5 * (1 + math.cos(math.pi * min(progress, 1.0)))
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr
        return lr

In [ ]:
NUM_EPOCHS = 100
WARMUP_STEPS = 500
GRAD_CLIP = 1.0
LABEL_SMOOTHING = 0.1
PATIENCE = 10

criterion = LabelSmoothingLoss(sp_en.get_piece_size(), padding_idx=PAD_IDX, smoothing=LABEL_SMOOTHING)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, betas=(0.9, 0.98), eps=1e-9, weight_decay=0.01)

total_steps = NUM_EPOCHS * len(train_loader)
scheduler = WarmupCosineScheduler(optimizer, D_MODEL, warmup_steps=WARMUP_STEPS, max_steps=total_steps)

print(f"Total training steps: {total_steps}")
print(f"Warmup steps: {WARMUP_STEPS}")
print(f"Peak LR: {scheduler.peak_lr:.6f}")
print(f"Early stopping patience: {PATIENCE}")

In [ ]:
def train_epoch(model, loader, criterion, optimizer, scheduler, grad_clip):
    model.train()
    total_loss = 0
    num_batches = 0

    for src, tgt in tqdm(loader, desc="Training", leave=False):
        src, tgt = src.to(device), tgt.to(device)
        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]

        logits = model(src, tgt_input)
        loss = criterion(logits, tgt_output)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        num_batches += 1

    return total_loss / num_batches


def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    num_batches = 0

    with torch.no_grad():
        for src, tgt in tqdm(loader, desc="Evaluating", leave=False):
            src, tgt = src.to(device), tgt.to(device)
            tgt_input = tgt[:, :-1]
            tgt_output = tgt[:, 1:]

            logits = model(src, tgt_input)
            loss = criterion(logits, tgt_output)

            total_loss += loss.item()
            num_batches += 1

    return total_loss / num_batches

In [ ]:
train_losses = []
dev_losses = []
best_dev_loss = float('inf')
patience_counter = 0

os.makedirs("checkpoints", exist_ok=True)

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_epoch(model, train_loader, criterion, optimizer, scheduler, GRAD_CLIP)
    dev_loss = evaluate(model, dev_loader, criterion)

    train_losses.append(train_loss)
    dev_losses.append(dev_loss)

    print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Dev Loss: {dev_loss:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")

    if dev_loss < best_dev_loss:
        best_dev_loss = dev_loss
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'dev_loss': dev_loss,
        }, "checkpoints/best_model.pt")
        print(f"  -> Saved best model (dev_loss: {dev_loss:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch} (no improvement for {PATIENCE} epochs)")
            break

print(f"\nBest dev loss: {best_dev_loss:.4f}")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss')
plt.plot(dev_losses, label='Dev Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss Curves')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

## 7. Beam Search Decoding

In [ ]:
def beam_search(model, src, beam_size=5, max_len=128, length_penalty=0.6):
    model.eval()
    with torch.no_grad():
        src = src.unsqueeze(0).to(device) if src.dim() == 1 else src.to(device)
        enc_out, src_mask = model.encode(src)

        beams = [(torch.tensor([BOS_IDX], device=device), 0.0)]
        completed = []

        for _ in range(max_len):
            candidates = []
            for seq, score in beams:
                if seq[-1].item() == EOS_IDX:
                    completed.append((seq, score))
                    continue

                tgt_input = seq.unsqueeze(0)
                logits = model.decode_step(tgt_input, enc_out, src_mask)
                log_probs = F.log_softmax(logits, dim=-1).squeeze(0)
                topk_probs, topk_ids = log_probs.topk(beam_size)

                for i in range(beam_size):
                    new_seq = torch.cat([seq, topk_ids[i:i+1]])
                    new_score = score + topk_probs[i].item()
                    candidates.append((new_seq, new_score))

            if not candidates:
                break

            candidates.sort(key=lambda x: x[1] / (len(x[0]) ** length_penalty), reverse=True)
            beams = candidates[:beam_size]

            if all(seq[-1].item() == EOS_IDX for seq, _ in beams):
                completed.extend(beams)
                break

        if not completed:
            completed = beams

        completed.sort(key=lambda x: x[1] / (len(x[0]) ** length_penalty), reverse=True)
        best_seq = completed[0][0]

        tokens = best_seq.tolist()
        if tokens[0] == BOS_IDX:
            tokens = tokens[1:]
        if EOS_IDX in tokens:
            tokens = tokens[:tokens.index(EOS_IDX)]

        return tokens


def greedy_decode(model, src, max_len=128):
    """Faster greedy decoding for efficiency."""
    model.eval()
    with torch.no_grad():
        src = src.unsqueeze(0).to(device) if src.dim() == 1 else src.to(device)
        enc_out, src_mask = model.encode(src)
        tgt_ids = [BOS_IDX]

        for _ in range(max_len):
            tgt_tensor = torch.tensor([tgt_ids], device=device)
            logits = model.decode_step(tgt_tensor, enc_out, src_mask)
            next_token = logits.argmax(dim=-1).item()
            if next_token == EOS_IDX:
                break
            tgt_ids.append(next_token)

        return tgt_ids[1:]

In [ ]:
checkpoint = torch.load("checkpoints/best_model.pt", map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best model from epoch {checkpoint['epoch']} with dev loss {checkpoint['dev_loss']:.4f}")

## 8. Translation & Evaluation

In [ ]:
def translate_sentence(model, sentence, sp_src, sp_tgt, beam_size=5):
    src_ids = [BOS_IDX] + sp_src.encode(sentence) + [EOS_IDX]
    src_tensor = torch.tensor(src_ids, dtype=torch.long)
    if beam_size > 1:
        output_ids = beam_search(model, src_tensor, beam_size=beam_size)
    else:
        output_ids = greedy_decode(model, src_tensor)
    return sp_tgt.decode(output_ids)


def translate_dataset(model, df, src_col, sp_src, sp_tgt, beam_size=5):
    translations = []
    model.eval()
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Translating"):
        src_sent = str(row[src_col]).strip()
        if src_sent and src_sent != 'nan':
            translation = translate_sentence(model, src_sent, sp_src, sp_tgt, beam_size)
        else:
            translation = ""
        translations.append(translation)
    return translations

In [ ]:
print("Translating dev set...")
dev_predictions = translate_dataset(model, dev_df, sa_col, sp_sa, sp_en, beam_size=5)

print("\nSample translations (Dev):")
for i in range(min(5, len(dev_df))):
    print(f"\n--- Example {i+1} ---")
    print(f"Source (SA): {dev_df[sa_col].iloc[i]}")
    print(f"Reference (EN): {dev_df[en_col].iloc[i]}")
    print(f"Predicted (EN): {dev_predictions[i]}")

## 9. Evaluation Metrics

In [ ]:
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

def compute_bleu(predictions, references):
    """Compute BLEU score using default NLTK BLEU with no weights."""
    refs = [[ref.split()] for ref in references]
    hyps = [pred.split() for pred in predictions]
    score = corpus_bleu(refs, hyps)
    return score


def compute_bert_score(predictions, references):
    """Compute BERTScore F1 with rescale_with_baseline=True."""
    from bert_score import score as bert_score_fn
    P, R, F1 = bert_score_fn(
        predictions, references,
        lang="en",
        rescale_with_baseline=True,
        verbose=True
    )
    return F1.mean().item()

In [ ]:
dev_references = [str(s).strip() for s in dev_df[en_col].tolist()]

dev_bleu = compute_bleu(dev_predictions, dev_references)
print(f"Dev BLEU Score: {dev_bleu:.4f}")

dev_bert_f1 = compute_bert_score(dev_predictions, dev_references)
print(f"Dev BERTScore F1: {dev_bert_f1:.4f}")

## 10. Test Set Translation & Submission

In [ ]:
print("Translating test set with beam search...")
start_time = time.time()
test_predictions = translate_dataset(model, test_df, sa_col, sp_sa, sp_en, beam_size=5)
inference_time = time.time() - start_time

print(f"\nInference time for test set: {inference_time:.2f} seconds")
print(f"Total parameters: {total_params:,}")

In [ ]:
test_references = [str(s).strip() for s in test_df[en_col].tolist()]

test_bleu = compute_bleu(test_predictions, test_references)
print(f"Test BLEU Score: {test_bleu:.4f}")

test_bert_f1 = compute_bert_score(test_predictions, test_references)
print(f"Test BERTScore F1: {test_bert_f1:.4f}")

print(f"\n{'='*50}")
print(f"SUMMARY OF RESULTS")
print(f"{'='*50}")
print(f"Dev  BLEU: {dev_bleu:.4f}")
print(f"Dev  BERTScore F1: {dev_bert_f1:.4f}")
print(f"Test BLEU: {test_bleu:.4f}")
print(f"Test BERTScore F1: {test_bert_f1:.4f}")
print(f"Inference Time: {inference_time:.2f} seconds")
print(f"Total Parameters: {total_params:,}")
print(f"{'='*50}")

In [ ]:
submission = pd.DataFrame({
    'Source_id': test_df['Source_id'].values,
    'Sentence_en': test_predictions
})

submission.to_csv('submission.csv', index=False, encoding='utf-8')
print("submission.csv generated successfully!")
print(f"\nFirst 5 rows:")
print(submission.head())

## 11. Efficiency Metrics

In [ ]:
print("="*50)
print("EFFICIENCY METRICS")
print("="*50)
print(f"Total inference time (test set): {inference_time:.2f} seconds")
print(f"Number of test sentences: {len(test_df)}")
print(f"Average time per sentence: {inference_time/len(test_df):.4f} seconds")
print(f"Total model parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print("="*50)

## 12. Translation Examples with Error Analysis

In [ ]:
print("="*70)
print("TRANSLATION EXAMPLES WITH ERROR ANALYSIS")
print("="*70)

num_examples = min(10, len(test_df))
indices = random.sample(range(len(test_df)), num_examples)

for idx in indices:
    src_sent = str(test_df[sa_col].iloc[idx]).strip()
    ref_sent = str(test_df[en_col].iloc[idx]).strip()
    pred_sent = test_predictions[idx]

    sent_bleu = sentence_bleu([ref_sent.split()], pred_sent.split())

    print(f"\n--- Example (Source_id: {test_df['Source_id'].iloc[idx]}) ---")
    print(f"Source (Sanskrit): {src_sent}")
    print(f"Reference (English): {ref_sent}")
    print(f"Prediction (English): {pred_sent}")
    print(f"Sentence BLEU: {sent_bleu:.4f}")

    ref_words = set(ref_sent.lower().split())
    pred_words = set(pred_sent.lower().split())
    missing = ref_words - pred_words
    extra = pred_words - ref_words
    if missing:
        print(f"Missing words: {missing}")
    if extra:
        print(f"Extra words: {extra}")
    print("-"*70)

## 13. Inference on Private Test Set

This cell can be used during evaluation to generate predictions on the private test set.

In [ ]:
def generate_submission_for_private_test(private_test_sa_path, output_path="submission.csv"):
    """Generate submission for private test set released during evaluation."""
    private_test_sa = pd.read_csv(private_test_sa_path)
    sa_column = [c for c in private_test_sa.columns if 'sa' in c.lower() and 'source' not in c.lower()][0]

    print(f"Private test set size: {len(private_test_sa)}")

    start_time = time.time()
    predictions = []
    model.eval()
    for _, row in tqdm(private_test_sa.iterrows(), total=len(private_test_sa), desc="Translating private test"):
        src_sent = str(row[sa_column]).strip()
        if src_sent and src_sent != 'nan':
            translation = translate_sentence(model, src_sent, sp_sa, sp_en, beam_size=5)
        else:
            translation = ""
        predictions.append(translation)
    inference_time = time.time() - start_time

    submission = pd.DataFrame({
        'Source_id': private_test_sa['Source_id'].values,
        'Sentence_en': predictions
    })
    submission.to_csv(output_path, index=False, encoding='utf-8')

    print(f"\nInference time: {inference_time:.2f} seconds")
    print(f"Total parameters: {total_params:,}")
    print(f"Submission saved to: {output_path}")
    return submission

# Uncomment and set the path when private test set is released:
# private_submission = generate_submission_for_private_test("data/private_test_sa.csv")

## 14. Enhanced Translation with Pre-trained NLLB Model

We fine-tune `facebook/nllb-200-distilled-600M`, a pre-trained multilingual seq2seq model that natively supports Sanskrit (`san_Deva`) and English (`eng_Latn`), on the provided 10K training pairs.

**Pre-trained model disclosure:** The NLLB-200-distilled-600M model (615M parameters) is used as a pre-trained backbone. It is fine-tuned exclusively on the provided training dataset. No external data or APIs are used.

In [ ]:
!pip install transformers accelerate -q

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

NLLB_MODEL_NAME = "facebook/nllb-200-distilled-600M"
nllb_tokenizer = AutoTokenizer.from_pretrained(NLLB_MODEL_NAME)
nllb_model = AutoModelForSeq2SeqLM.from_pretrained(NLLB_MODEL_NAME).to(device)

nllb_params = sum(p.numel() for p in nllb_model.parameters())
print(f"NLLB parameters: {nllb_params:,}")
print(f"Device: {device}")

In [ ]:
class NLLBTranslationDataset(Dataset):
    def __init__(self, df, tokenizer, src_col, tgt_col, max_len=128):
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.pairs = []
        for _, row in df.iterrows():
            src = str(row[src_col]).strip()
            tgt = str(row[tgt_col]).strip()
            if src and tgt and src != 'nan' and tgt != 'nan':
                self.pairs.append((src, tgt))

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src, tgt = self.pairs[idx]
        self.tokenizer.src_lang = "san_Deva"
        model_inputs = self.tokenizer(
            src, max_length=self.max_len, truncation=True,
            padding="max_length", return_tensors="pt"
        )
        labels = self.tokenizer(
            text_target=tgt, max_length=self.max_len, truncation=True,
            padding="max_length", return_tensors="pt"
        )
        label_ids = labels["input_ids"].squeeze(0)
        label_ids[label_ids == self.tokenizer.pad_token_id] = -100
        model_inputs["labels"] = label_ids
        return {k: v.squeeze(0) for k, v in model_inputs.items()}


NLLB_BATCH_SIZE = 16
nllb_train_dataset = NLLBTranslationDataset(train_df, nllb_tokenizer, sa_col, en_col)
nllb_dev_dataset = NLLBTranslationDataset(dev_df, nllb_tokenizer, sa_col, en_col)

nllb_train_loader = DataLoader(nllb_train_dataset, batch_size=NLLB_BATCH_SIZE, shuffle=True)
nllb_dev_loader = DataLoader(nllb_dev_dataset, batch_size=NLLB_BATCH_SIZE, shuffle=False)

print(f"NLLB train batches: {len(nllb_train_loader)}, dev batches: {len(nllb_dev_loader)}")

In [ ]:
NLLB_EPOCHS = 5
NLLB_LR = 2e-5
NLLB_GRAD_CLIP = 1.0

for param in nllb_model.model.shared.parameters():
    param.requires_grad = False

nllb_optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, nllb_model.parameters()),
    lr=NLLB_LR, weight_decay=0.01
)

nllb_train_losses = []
nllb_dev_losses = []
best_nllb_dev_loss = float('inf')

os.makedirs("checkpoints", exist_ok=True)

for epoch in range(1, NLLB_EPOCHS + 1):
    nllb_model.train()
    total_loss = 0
    num_batches = 0
    for batch in tqdm(nllb_train_loader, desc=f"NLLB Epoch {epoch}", leave=False):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = nllb_model(**batch)
        loss = outputs.loss

        nllb_optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(nllb_model.parameters(), NLLB_GRAD_CLIP)
        nllb_optimizer.step()

        total_loss += loss.item()
        num_batches += 1

    avg_train_loss = total_loss / num_batches
    nllb_train_losses.append(avg_train_loss)

    nllb_model.eval()
    dev_total = 0
    dev_batches = 0
    with torch.no_grad():
        for batch in tqdm(nllb_dev_loader, desc="NLLB Dev", leave=False):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = nllb_model(**batch)
            dev_total += outputs.loss.item()
            dev_batches += 1

    avg_dev_loss = dev_total / dev_batches
    nllb_dev_losses.append(avg_dev_loss)

    print(f"NLLB Epoch {epoch} | Train Loss: {avg_train_loss:.4f} | Dev Loss: {avg_dev_loss:.4f}")

    if avg_dev_loss < best_nllb_dev_loss:
        best_nllb_dev_loss = avg_dev_loss
        torch.save(nllb_model.state_dict(), "checkpoints/best_nllb.pt")
        print(f"  -> Saved best NLLB model (dev_loss: {avg_dev_loss:.4f})")

print(f"\nBest NLLB dev loss: {best_nllb_dev_loss:.4f}")

In [ ]:
nllb_model.load_state_dict(torch.load("checkpoints/best_nllb.pt", map_location=device))
nllb_model.eval()
print("Loaded best NLLB checkpoint")


def translate_nllb(model, tokenizer, sentences, batch_size=32, max_len=128, num_beams=4):
    model.eval()
    tokenizer.src_lang = "san_Deva"
    tgt_lang_id = tokenizer.convert_tokens_to_ids("eng_Latn")
    all_translations = []

    for i in tqdm(range(0, len(sentences), batch_size), desc="NLLB Translating"):
        batch = sentences[i:i + batch_size]
        inputs = tokenizer(
            batch, return_tensors="pt", padding=True,
            truncation=True, max_length=max_len
        ).to(device)

        with torch.no_grad():
            generated = model.generate(
                **inputs,
                forced_bos_token_id=tgt_lang_id,
                max_new_tokens=max_len,
                num_beams=num_beams
            )

        translations = tokenizer.batch_decode(generated, skip_special_tokens=True)
        all_translations.extend(translations)

    return all_translations


dev_sentences = [str(s).strip() for s in dev_df[sa_col].tolist()]
nllb_dev_predictions = translate_nllb(nllb_model, nllb_tokenizer, dev_sentences)

print("\nSample NLLB translations (Dev):")
for i in range(min(5, len(dev_df))):
    print(f"\n--- Example {i+1} ---")
    print(f"Source (SA): {dev_df[sa_col].iloc[i]}")
    print(f"Reference (EN): {dev_df[en_col].iloc[i]}")
    print(f"NLLB Predicted (EN): {nllb_dev_predictions[i]}")

In [ ]:
nllb_dev_references = [str(s).strip() for s in dev_df[en_col].tolist()]

nllb_dev_bleu = compute_bleu(nllb_dev_predictions, nllb_dev_references)
print(f"NLLB Dev BLEU Score: {nllb_dev_bleu:.4f}")

nllb_dev_bert_f1 = compute_bert_score(nllb_dev_predictions, nllb_dev_references)
print(f"NLLB Dev BERTScore F1: {nllb_dev_bert_f1:.4f}")

In [ ]:
print("Translating test set with NLLB...")
test_sentences = [str(s).strip() for s in test_df[sa_col].tolist()]

start_time = time.time()
nllb_test_predictions = translate_nllb(nllb_model, nllb_tokenizer, test_sentences)
nllb_inference_time = time.time() - start_time

print(f"\nNLLB Inference time for test set: {nllb_inference_time:.2f} seconds")
print(f"NLLB Total parameters: {nllb_params:,}")

In [ ]:
nllb_test_references = [str(s).strip() for s in test_df[en_col].tolist()]

nllb_test_bleu = compute_bleu(nllb_test_predictions, nllb_test_references)
print(f"NLLB Test BLEU Score: {nllb_test_bleu:.4f}")

nllb_test_bert_f1 = compute_bert_score(nllb_test_predictions, nllb_test_references)
print(f"NLLB Test BERTScore F1: {nllb_test_bert_f1:.4f}")

print(f"\n{'='*60}")
print(f"NLLB SUMMARY OF RESULTS")
print(f"{'='*60}")
print(f"NLLB Dev  BLEU:        {nllb_dev_bleu:.4f}")
print(f"NLLB Dev  BERTScore:   {nllb_dev_bert_f1:.4f}")
print(f"NLLB Test BLEU:        {nllb_test_bleu:.4f}")
print(f"NLLB Test BERTScore:   {nllb_test_bert_f1:.4f}")
print(f"NLLB Inference Time:   {nllb_inference_time:.2f} seconds")
print(f"NLLB Total Parameters: {nllb_params:,}")
print(f"{'='*60}")

In [ ]:
nllb_submission = pd.DataFrame({
    'Source_id': test_df['Source_id'].values,
    'Sentence_en': nllb_test_predictions
})

nllb_submission.to_csv('submission.csv', index=False, encoding='utf-8')
print("submission.csv generated with NLLB predictions!")
print(f"\nFirst 5 rows:")
print(nllb_submission.head())

## 15. NLLB Inference on Private Test Set

Uncomment and run during evaluation when the private test set is released.

In [ ]:
def generate_nllb_submission_for_private_test(private_test_sa_path, output_path="submission.csv"):
    """Generate submission for private test set using NLLB model."""
    private_test_sa = pd.read_csv(private_test_sa_path)
    sa_column = [c for c in private_test_sa.columns if c.endswith('_sa') and 'source' not in c.lower()][0]

    print(f"Private test set size: {len(private_test_sa)}")

    sentences = [str(row[sa_column]).strip() for _, row in private_test_sa.iterrows()]

    start_time = time.time()
    predictions = translate_nllb(nllb_model, nllb_tokenizer, sentences, batch_size=32, num_beams=4)
    inf_time = time.time() - start_time

    submission = pd.DataFrame({
        'Source_id': private_test_sa['Source_id'].values,
        'Sentence_en': predictions
    })
    submission.to_csv(output_path, index=False, encoding='utf-8')

    print(f"\nInference time: {inf_time:.2f} seconds")
    print(f"Total parameters: {nllb_params:,}")
    print(f"Submission saved to: {output_path}")
    return submission

# Uncomment and set the path when private test set is released:
# private_submission = generate_nllb_submission_for_private_test("data/private_test_sa.csv")